# Getting Started with IPAM Tool Development

Welcome to the Firewall Automation Hackathon! This notebook will guide you through building the IPAM (IP Address Management) validation tools.

## Overview

The IPAM tools provide IP address validation and conflict detection for firewall rules. These tools will be integrated into the supervisor agent. The tools:
- Validate IP address and CIDR syntax
- Check if IP/CIDR is already allocated in your-ipam-server.example.com
- Detect conflicts with existing IP allocations
- Classify IPs (private, public, reserved)
- Verify CIDR availability before rule creation
- Provide security recommendations based on IP type

## Related Jira Task

- [FWAUTO-21: Implement Firewall IPAM Agent (Phase 2/3)](https://your-jira-instance.atlassian.net/browse/FWAUTO-21)

## How to Get Started

Your task is to implement **IPAM validation tools** that query the your organization IPAM table at **your-ipam-server.example.com** and will be added to the supervisor agent in `agent/src/agent.py`.

### IPAM Integration

Query the your organization IPAM table at **your-ipam-server.example.com** for:
- IP/CIDR allocation status
- Owner and account information
- Conflict detection with existing allocations
- Regional and environment metadata

### Credentials

Setup IPAM credentials (API keys) in Secrets Manager and pass it to AgentCore when Authenticating to the IPAM service.

### If you have time...

Explore Bedrock AgentCore Gateway to see if you can convert the IPAM API into a MCP server.

### Key Libraries

- **ipaddress**: Python built-in module for IP validation and classification
- **requests** or **boto3**: For querying your-ipam-server.example.com (depending on API/database type)
- **datetime**: For cache TTL management (10-minute TTL recommended)

### Reference Implementation

Study the `network_firewall_analyser_agent.py` in the firewall-logs-agent folder for:
- Tool definitions using `@tool` decorator
- Error handling and result formatting

## Step 1: Copy the Base Agent Code

Copy the supervisor agent code to your workspace so you can modify it.

In [1]:
%%bash
# Copy the base agent code from the repository
cp -r /home/sakhan/IST-AWS-Firewall-Automation/agent .

# Remove the existing bedrock_agentcore.yaml configuration
rm -f agent/.bedrock_agentcore.yaml

In [2]:
# View the base agent python code
with open('agent/src/agent.py', 'r') as f:
    print(f.read())

# Firewall Automation Supervisor Agent
# To test locally, run `uv run agent.py` and then
# curl -X POST http://localhost:8080/invocations -H "Content-Type: application/json" -d '{"prompt": "Show me firewall logs for account A123"}'

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

app = BedrockAgentCoreApp()


@tool
def query_account_details(account_identifier: str):
    """
    Query AWS account details from DynamoDB.

    Args:
        account_identifier: Account name, ID, or CIDR range to search for

    Returns:
        Account details including name, ID, CIDR ranges, and metadata
    """
    return {
        "account_name": "Production-App-Account",
        "account_id": "123456789012",
        "cidr_ranges": ["10.100.0.0/16", "10.101.0.0/16"],
        "environment": "production",
        "owner": "Platform Team",
        "region": "ap-southeast-2"
    }


@tool
def search_firewall_logs(
    query: s

## Step 2: Review the README

Study the README.md file in this folder to understand the DynamoDB schema, tool definitions, and implementation patterns.

In [3]:
# View the README for implementation guidance
with open('README.md', 'r') as f:
    print(f.read())

# Firewall IPAM Agent Development

This folder contains resources for building the Firewall IPAM (IP Address Management) Agent.

## Overview

The IPAM Agent verifies IP address allocations and prevents conflicts:
- Check if IP/CIDR is already allocated
- Verify IP ownership
- Prevent IP conflicts in firewall rules
- Validate IP ranges before rule creation

## Related Jira Tasks

- **FWAUTO-21**: Implement Firewall IPAM Agent (Phase 2/3)

## Getting Started

This folder doesn't have sample notebooks yet. Reference the firewall-logs-tool and account-details-agent for similar integration patterns.

## Key Components to Build

### Tool Definitions

```python
@tool
def check_ip_allocation(ip_or_cidr: str) -> dict:
    """
    Check if an IP or CIDR is allocated and to whom

    Args:
        ip_or_cidr: IP address (192.168.1.1) or CIDR (10.0.0.0/24)

    Returns:
        {
            "allocated": bool,
            "owner": str or None,
            "account_id": str or None,
            "vp

## Step 3: Implement IPAM Validation Tools

Replace the dummy `validate_ip_addresses` tool in `agent/src/agent.py` with real IPAM validation functionality.

### Key Implementation Steps:

1. **Set up IPAM connection** - Connect to your-ipam-server.example.com (determine API/database access method)
2. **Implement validation tools** - Replace the dummy tool with real functionality:
   - Validate CIDR syntax using Python's ipaddress module
   - Query your-ipam-server.example.com for IP/CIDR allocation status
   - Detect conflicts with existing allocations
   - Classify IP types (private, public, reserved)
3. **Add conflict detection** - Check for overlapping CIDR ranges
4. **Implement caching** - Add in-memory cache with 10-minute TTL
5. **Add error handling** - Handle IPAM service errors, invalid inputs, network failures

### IP Validation with ipaddress Module:

```python
import ipaddress

def validate_cidr_syntax(cidr: str) -> dict:
    """Validate CIDR notation syntax"""
    try:
        network = ipaddress.ip_network(cidr, strict=False)
        return {
            "valid": True,
            "network_address": str(network.network_address),
            "broadcast_address": str(network.broadcast_address),
            "num_addresses": network.num_addresses,
            "is_private": network.is_private,
            "error": None
        }
    except ValueError as e:
        return {"valid": False, "error": str(e)}

def check_overlap(cidr1: str, cidr2: str) -> bool:
    """Check if two CIDR ranges overlap"""
    net1 = ipaddress.ip_network(cidr1)
    net2 = ipaddress.ip_network(cidr2)
    return net1.overlaps(net2)
```

### IPAM Query Example:

Query your-ipam-server.example.com for allocation details:
- Check if CIDR is allocated
- Get owner and account information
- Retrieve regional metadata
- Identify conflicts with existing allocations

### Implementation Approach:

These are **tools for the supervisor agent** to validate IP addresses:
- Uses ipaddress module for syntax validation and overlap detection
- Queries your-ipam-server.example.com for allocation status
- Caches results to reduce IPAM queries (10-minute TTL)
- Provides security recommendations based on IP type
- The supervisor agent will use these tools before creating firewall rules

In [4]:
# View the current dummy implementation
with open('agent/src/agent.py', 'r') as f:
    content = f.read()
    # Find and display the validate_ip_addresses function
    start = content.find('def validate_ip_addresses')
    end = content.find('\n\n\nmodel_id', start)
    if start != -1:
        print(content[start:end if end != -1 else start+1000])

def validate_ip_addresses(
    cidr_range: str = None,
    ip_address: str = None,
    check_conflicts: bool = True
):
    """
    Validate IP addresses and CIDR ranges using AWS VPC IPAM.

    Args:
        cidr_range: CIDR range to validate
        ip_address: Single IP address to validate
        check_conflicts: Check for conflicts with existing ranges

    Returns:
        Validation results and conflict information
    """
    return {
        "valid": True,
        "cidr": cidr_range or ip_address,
        "type": "private" if cidr_range and cidr_range.startswith("10.") else "public",
        "conflicts": [],
        "available": True,
        "message": "CIDR range is valid and available with no conflicts"
    }


## Step 4: Test Your Implementation

Test the tool locally before deploying.

In [ ]:
import base64
import requests

username = 'mlops-test'
password = 'xxxx'
SOLIDSERVER_URL = "https://your-ipam-server.example.com"

# Encode credentials
HEADERS = {
    'x-ipm-username': base64.b64encode(username.encode()).decode(),
    'x-ipm-password': base64.b64encode(password.encode()).decode(),
    'cache-control': 'no-cache'
}


In [5]:
import ipaddress
def calculate_subnet_bounds(subnet_cidr: str) -> dict:
    """
    Calculates start and end IP addresses of a given subnet.

    Args:
        subnet_cidr: CIDR format (e.g., '10.9.56.0/24')

    Returns:
        Dict with network, broadcast, and usable range
    """
    network = ipaddress.ip_network(subnet_cidr, strict=False)
    return {
        "start_ip_addr": str(network.network_address),
        "end_ip_addr": str(network.broadcast_address),
        "first_usable": str(list(network.hosts())[0]) if network.num_addresses > 2 else str(network.network_address),
        "last_usable": str(list(network.hosts())[-1]) if network.num_addresses > 2 else str(network.broadcast_address),
        "total_addresses": network.num_addresses
    }

In [ ]:
import ipaddress
from urllib.parse import unquote

def describe_subnet_owner_region(subnet_object: dict, cidr_range: str = None, ipaddress: str = None) -> str:
    """
    Returns a formatted string describing the subnet CIDR, owner, and region.

    Args:
        subnet_object: A dictionary from EfficientIP IPAM response containing subnet details.
        cidr_range: The CIDR range string (e.g., '10.9.56.0/24')
        ipaddress: An individual IP address string (e.g., '10.9.56.1')

    Returns:
        A human-readable string like:
        "The CIDR range 10.9.56.0/24 is owned by XYZ and is in the region ABC."
    """

    params_raw = subnet_object.get("subnet_class_parameters", "")
    params_decoded = unquote(params_raw)
    params = dict(item.split("=", 1) for item in params_decoded.split("&") if "=" in item)

    owner = params.get("owner", "Unknown")
    region = params.get("region", "Unknown")
    if cidr_range:
        return f"The CIDR range {cidr_range} is owned by {owner} and is in the region {region}."
    elif ipaddress:
        return f"The IP address {ipaddress} is owned by {owner} and is in the region {region}."


In [ ]:
def check_cidr_range(cidr_range: str) -> dict:
    """
    Check if a specific cidr_range  already exists in IPAM.

    Args:
        cidr_range: CIDR range to validate

    Returns:
        True if cidr_range exists, False otherwise
    """
    url = f"{SOLIDSERVER_URL}/rest/ip_block_subnet_list"
    subnet_bound_response = calculate_subnet_bounds(cidr_range)
    start_ip_addr = subnet_bound_response.get('start_ip_addr')
    end_ip_addr = subnet_bound_response.get('end_ip_addr')
    params = {
        "WHERE": f"start_hostaddr='{start_ip_addr}' and end_hostaddr='{end_ip_addr}'"
    }


    response = requests.get(url, headers=HEADERS, params=params, verify=False)
    
    if response.ok:
        if response.text:
            data = response.json()
            return {"status": "success", "exists": True, "message": describe_subnet_owner_region(data[0], cidr_range=cidr_range)}
        else:
            return {"status": "success", "exists": False, "message": f"The CIDR range {cidr_range} entry does not exist in IPAM"}
    return {"status": "error", "error": response.text}

In [50]:
check_cidr_range("10.10.128.24/30")

/home/sakhan/IST-AWS-Firewall-Automation/app/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'your-ipam-server.example.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'status': 'success',
 'exists': True,
 'message': 'The CIDR range 10.10.128.24/30 is owned by Global Network Team and is in the region Global.'}

In [37]:
def check_ip_addresses(ip_address: str = None) -> dict:
    """
    Validate if the given IP address is available in IPAM.

    Args:
        ip_address: IP address to validate (e.g., '10.10.128.24')
    """
    url = f"{SOLIDSERVER_URL}/rest/ip_address_list"
    params = {}
    if ip_address:
        params["WHERE"] = f"hostaddr='{ip_address}'"
    response = requests.get(url, headers=HEADERS, params=params, verify=False)
    if response.ok:
        if response.text:
            data = response.json()
            return {"status": "success", "exists": True, "message": describe_subnet_owner_region(data[0], ipaddress=ip_address)}
        else:
            return {"status": "success", "exists": False, "message": f"{ip_address} entry does not exist in IPAM"}
    return {"status": "error", "error": response.text}

In [39]:
check_ip_addresses("10.10.128.26")

/home/sakhan/IST-AWS-Firewall-Automation/app/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'your-ipam-server.example.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'status': 'success',
 'exists': False,
 'message': '10.10.128.26 entry does not exist in IPAM'}

In [45]:
import ipaddress

def classify_ip_or_cidr(value: str) -> str:
    """
    Classifies whether the input IP or CIDR block is private or public.

    Args:
        value: A string representing an IP address or a CIDR block (e.g., '10.0.0.1', '192.168.1.0/24')

    Returns:
        One of: 'private', 'public' or 'invalid'
    """
    try:
        # Try parsing as a network (CIDR block)
        network = ipaddress.ip_network(value, strict=False)
        if network.is_private:
            return "private"
        else:
            return "public"
    except ValueError:
        try:
            # Fallback: try parsing as a single IP
            ip = ipaddress.ip_address(value)
            if ip.is_private:
                return "private"
            else:
                return "public"
        except ValueError:
            return "invalid"


In [51]:
def validate_ip_addresses(
    ip_address: str = None,
    cidr_range: str = None,
):
    """
    Validate IP addresses and CIDR ranges using AWS VPC IPAM.

    Args:
        ip_address: Single IP address to validate
        cidr_range: CIDR range to validate

    Returns:
        Validation results and conflict information
    """
    if cidr_range:
        validation_result = check_cidr_range(cidr_range)
        if validation_result.get("status") == "error":
            valid = False
            message = f"Error validating CIDR range: {validation_result.get('error')}"
        if validation_result.get("exists"):
            valid = False
            message = f"CIDR range conflict: {validation_result.get('message')}"
        else:
            valid = True
            message = f"CIDR range {cidr_range} is valid and available with no conflicts"
    elif ip_address:
        validation_result = check_ip_addresses(ip_address)
        if validation_result.get("status") == "error":
            valid = False
            message = f"Error validating IP address: {validation_result.get('error')}"
        if validation_result.get("exists"):
            valid = False
            message = f"IP address conflict: {validation_result.get('message')}"
        else:
            valid = True
            message = f"IP address {ip_address} is valid and available with no conflicts"
    return {
        "valid": valid,
        "cidr": cidr_range or ip_address,
        "type": classify_ip_or_cidr(cidr_range or ip_address),
        "available": valid,
        "message": message
    }

In [52]:
validate_ip_addresses(cidr_range="10.10.128.24/30")

/home/sakhan/IST-AWS-Firewall-Automation/app/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'your-ipam-server.example.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'valid': False,
 'cidr': '10.10.128.24/30',
 'type': 'private',
 'available': False,
 'message': 'CIDR range conflict: The CIDR range 10.10.128.24/30 is owned by Global Network Team and is in the region Global.'}

In [53]:
validate_ip_addresses(ip_address="10.10.128.26")

/home/sakhan/IST-AWS-Firewall-Automation/app/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'your-ipam-server.example.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'valid': True,
 'cidr': '10.10.128.26',
 'type': 'private',
 'available': True,
 'message': 'IP address 10.10.128.26 is valid and available with no conflicts'}

In [ ]:
# TODO: Add your test code here
# Example test:
# from agent.src.agent import validate_ip_addresses
# 
# # Test CIDR validation
# result = validate_ip_addresses(cidr_range="10.10.0.0/16", check_conflicts=True)
# print(f"CIDR validation result:\n{result}")
# 
# # Test IP address validation
# result = validate_ip_addresses(ip_address="192.168.1.1", check_conflicts=False)
# print(f"IP validation result:\n{result}")
# 
# # Test conflict detection
# result = validate_ip_addresses(cidr_range="10.10.5.0/24", check_conflicts=True)
# print(f"Conflict detection result:\n{result}")

## Step 5: Deploy to AWS

Deploy your updated supervisor agent (with the new IPAM validation tools) to AWS Bedrock AgentCore Runtime.

In [ ]:
import time
import boto3
from bedrock_agentcore_starter_toolkit import Runtime

# TODO: Set your agent name
agent_name = <AGENT_NAME>  # e.g., "firewall-supervisor-agent"

# Initialize the runtime toolkit
region = "ap-southeast-2"

agentcore_runtime = Runtime()

# Configure the deployment
response = agentcore_runtime.configure(
    agent_name=agent_name,
    entrypoint=<ENTRYPOINT>,  # TODO: Set your entrypoint file, e.g., "agent/src/agent.py"
    execution_role="arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole",
    code_build_execution_role="arn:aws:iam::123456789012:role/YourCodeBuildRole",
    auto_create_ecr=True,
    requirements_file=<REQUIREMENTS_FILE>,  # TODO: Set your requirements file, e.g., "agent/src/requirements.txt"
    region=region,
    memory_mode="STM_ONLY",
)

print("Configuration completed:", response)

launch_result = agentcore_runtime.launch()
print("Launch completed:", launch_result.agent_arn)

# Wait for the agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Waiting for deployment... Current status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

if status == "READY":
    runtime_id = status_response.agent["agentRuntimeId"]

    # Update the runtime to be deployed in VPC
    client = boto3.client("bedrock-agentcore-control", region_name=region)

    response = client.update_agent_runtime(
        agentRuntimeId=runtime_id,
        networkConfiguration={
            "networkMode": "VPC",
            "networkModeConfig": {
                "subnets": ["subnet-xxxxxxxxxxxxxxxxx", "subnet-yyyyyyyyyyyyyyyyy"],
                "securityGroups": ["sg-xxxxxxxxxxxxxxxxx"],
            },
        },
        agentRuntimeArtifact={
            "containerConfiguration": {
                "containerUri": f"123456789012.dkr.ecr.ap-southeast-2.amazonaws.com/bedrock-agentcore-{agent_name}"
            }
        },
        roleArn="arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole",
    )
    
print(f"Supervisor Agent deployed successfully with IPAM validation tools!")

## Step 6: Test End-to-End

Test the deployed agent through the Streamlit UI.

## Step 7: Push Your Changes

Create a branch and push your changes for review.

## Tips and Best Practices

### Security
- **Store IPAM credentials in AWS Secrets Manager**: Never hardcode credentials for your-ipam-server.example.com access
- **Validate all IP inputs**: Prevent injection attacks by validating syntax first
- **Read-only access**: IPAM tools should not allocate/deallocate IPs
- **Audit logging**: Log all IP validations for security audit trail
- **Private IP warnings**: Flag rules allowing access to private IPs from public sources

### IP Validation
- **Use ipaddress module**: Python's built-in module for robust validation
- **Syntax validation first**: Always validate CIDR syntax before querying IPAM
- **Handle IPv4 and IPv6**: Support both IP versions
- **Normalize inputs**: Strip whitespace, convert to standard format

### IPAM Queries
- **Connection to your-ipam-server.example.com**: Determine the API or database access method
- **Authentication**: Retrieve credentials from AWS Secrets Manager (e.g., `firewall-chatbot/ipam/credentials`)
- **Cache results**: 10-minute TTL (IPs change less frequently than other data)
- **Batch queries**: Query multiple CIDRs in a single request when possible

### Conflict Detection
- **Check overlaps**: Use ipaddress.ip_network.overlaps() for CIDR comparison
- **Aggregate ranges**: Pre-aggregate large CIDR blocks for efficiency
- **Provide suggestions**: Suggest alternative ranges when conflicts exist
- **Detail conflicts**: Show which specific allocations conflict

### Performance
- **Cache negative results**: Cache "IP not found" results to reduce queries
- **Lazy validation**: Only query IPAM when necessary (after syntax validation)
- **Index by account**: Pre-filter by account ID when available
- **Timeout handling**: Set reasonable timeouts for IPAM queries

### User Experience
- **Clear error messages**: "Invalid CIDR format: 10.10.0.0/33 (prefix must be 0-32)"
- **Security recommendations**: "Public IP detected - ensure proper justification for external access"
- **Conflict details**: "CIDR overlaps with 10.10.0.0/16 allocated to RT-Prod-DR"
- **Suggestions**: "Try more specific range: 10.10.5.0/25"

## Resources

- README: `README.md` in this folder
- Jira: [FWAUTO-21](https://your-jira-instance.atlassian.net/browse/FWAUTO-21)
- Python ipaddress module: https://docs.python.org/3/library/ipaddress.html
- AgentCore Gateway: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-core-concepts.html
- `./02-transform-apis-into-mcp-tools` on how to use existing API as MCP servers using Gateway
- `/home/sagemaker-user/amazon-bedrock-agentcore-samples/` for more examples from AWS